# Lab 10 · Reference solution

The polished final implementation of [Lab 10: Supervisor-worker from scratch](../README.md).

A supervisor coordinates a researcher worker (web search + fetch, with action-hash dedup) and a writer worker (citation-preserving prose). Step caps escalate cleanly across agent levels via structured-error envelopes. No frameworks.

This notebook is the reference implementation; refer to [`../lab.ipynb`](../lab.ipynb) for the pedagogical step-by-step build. The [`solution README`](./README.md) covers implementation choices, common variations, and bugs to watch for.

## Setup

In [ ]:
import hashlib
import json
import os
import pathlib
import re
import warnings
from dataclasses import dataclass, field
from typing import Any, Literal

from dotenv import load_dotenv
from pydantic import BaseModel, ConfigDict, Field

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY"), (
    "Set OPENAI_API_KEY or ANTHROPIC_API_KEY in .env"
)

PROVIDER = "openai"
MODEL = {"openai": "gpt-4o-mini", "anthropic": "claude-haiku-4-5-20251001"}[PROVIDER]
print(f"Using {PROVIDER} / {MODEL}")


## Provider-agnostic chat client

In [ ]:
@dataclass
class ToolCall:
    id: str
    name: str
    arguments: dict


@dataclass
class AssistantMessage:
    content: str | None
    tool_calls: list[ToolCall] = field(default_factory=list)


def chat_with_tools(messages: list[dict], tools: list[dict] | None = None,
                     tool_choice: str = "auto", temperature: float = 0) -> AssistantMessage:
    if PROVIDER == "openai":
        from openai import OpenAI
        resp = OpenAI().chat.completions.create(
            model=MODEL, messages=messages, tools=tools,
            tool_choice=tool_choice if tools else None, temperature=temperature,
        )
        msg = resp.choices[0].message
        return AssistantMessage(
            content=msg.content,
            tool_calls=[
                ToolCall(id=tc.id, name=tc.function.name,
                         arguments=json.loads(tc.function.arguments))
                for tc in (msg.tool_calls or [])
            ],
        )
    elif PROVIDER == "anthropic":
        from anthropic import Anthropic
        client = Anthropic()
        system = next((m["content"] for m in messages if m["role"] == "system"), "")
        non_system = [m for m in messages if m["role"] != "system"]
        anth_tools = [
            {"name": t["function"]["name"], "description": t["function"]["description"],
             "input_schema": t["function"]["parameters"]}
            for t in (tools or [])
        ]
        resp = client.messages.create(
            model=MODEL, system=system, messages=non_system,
            tools=anth_tools or None, max_tokens=2048, temperature=temperature,
        )
        text = "".join(b.text for b in resp.content if hasattr(b, "text"))
        tcs = [ToolCall(id=b.id, name=b.name, arguments=dict(b.input))
               for b in resp.content if getattr(b, "type", None) == "tool_use"]
        return AssistantMessage(content=text or None, tool_calls=tcs)
    raise ValueError(f"Unknown PROVIDER: {PROVIDER!r}")


def _action_hash(name: str, args: dict) -> str:
    """Structural action signature for dedup. Same as Lab 03."""
    return hashlib.sha256(
        (name + "|" + json.dumps(args, sort_keys=True)).encode()
    ).hexdigest()[:16]


class StrictModel(BaseModel):
    """Lab 02 pattern: extra='forbid' rejects LLM-invented fields."""
    model_config = ConfigDict(extra="forbid")


## Worker-level tools: `web_search` + `fetch_page`

In [ ]:
from ddgs import DDGS
from ddgs.exceptions import DDGSException, RatelimitException, TimeoutException
import requests
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning

warnings.simplefilter("ignore", MarkupResemblesLocatorWarning)

RecencyType = Literal["any", "day", "week", "month", "year"]
_RECENCY_MAP: dict[str, str | None] = {
    "any": None, "day": "d", "week": "w", "month": "m", "year": "y",
}
USER_AGENT = (
    "AgenticAIEngineer-CourseLab/0.1 "
    "(https://github.com/MHHamdan/Agentic-AI-Engineer)"
)
PAYWALL_MARKERS = [
    "subscribe to read", "subscribe to continue",
    "create a free account to continue",
    "you've reached your free article limit",
    "register to read",
]


def web_search(query: str, recency: RecencyType = "any", max_results: int = 8) -> dict:
    if not query or not query.strip():
        return {"status": "error", "kind": "other", "detail": "empty query"}
    try:
        with DDGS(timeout=15) as ddgs:
            raw = ddgs.text(
                query=query.strip(), region="us-en", safesearch="moderate",
                timelimit=_RECENCY_MAP.get(recency),
                max_results=max_results, backend="auto",
            )
    except RatelimitException as e:
        return {"status": "error", "kind": "rate_limit", "detail": str(e)}
    except TimeoutException as e:
        return {"status": "error", "kind": "timeout", "detail": str(e)}
    except DDGSException as e:
        return {"status": "error", "kind": "other", "detail": str(e)}
    except Exception as e:
        return {"status": "error", "kind": "other", "detail": f"{type(e).__name__}: {e}"}
    if not raw:
        return {"status": "empty", "query": query, "detail": "no results"}
    return {
        "status": "ok",
        "results": [
            {"title": (r.get("title") or "").strip(),
             "url": (r.get("href") or "").strip(),
             "snippet": (r.get("body") or "").strip()}
            for r in raw if r.get("href")
        ][:max_results],
    }


def fetch_page(url: str, max_chars: int = 8000) -> dict:
    if not url or not url.startswith(("http://", "https://")):
        return {"status": "error", "url": url, "kind": "other", "detail": "invalid url"}
    try:
        resp = requests.get(url, headers={"User-Agent": USER_AGENT},
                            timeout=15, allow_redirects=True)
    except requests.Timeout:
        return {"status": "error", "url": url, "kind": "timeout",
                "detail": "request timed out after 15s"}
    except requests.RequestException as e:
        return {"status": "error", "url": url, "kind": "other",
                "detail": f"{type(e).__name__}: {e}"}
    if 400 <= resp.status_code < 500:
        kind = "blocked" if resp.status_code in (401, 403, 429) else "http_4xx"
        return {"status": "error", "url": url, "kind": kind,
                "detail": f"HTTP {resp.status_code}"}
    if 500 <= resp.status_code < 600:
        return {"status": "error", "url": url, "kind": "http_5xx",
                "detail": f"HTTP {resp.status_code}"}
    try:
        soup = BeautifulSoup(resp.text, "html.parser")
    except Exception as e:
        return {"status": "error", "url": url, "kind": "parse",
                "detail": f"{type(e).__name__}: {e}"}
    for tag in soup(["script", "style", "nav", "footer", "aside",
                     "header", "form", "iframe", "noscript"]):
        tag.decompose()
    title = (soup.title.string.strip() if soup.title and soup.title.string else "")
    text = re.sub(r"\n{3,}", "\n\n", soup.get_text(separator="\n")).strip()
    if any(m in text[:5000].lower() for m in PAYWALL_MARKERS):
        return {"status": "error", "url": url, "kind": "paywall",
                "detail": "paywall markers detected"}
    if len(text) > max_chars:
        return {"status": "too_long", "url": url, "title": title,
                "text": text[:max_chars], "total_chars": len(text)}
    return {"status": "ok", "url": url, "title": title, "text": text}


## Researcher worker

Lab 03's research agent, repackaged as a single-shot worker. Takes a question, returns `{status, findings, citations}` on completion or `{status: "step_cap", ...}` if budget exhausted.

In [ ]:
WORKER_MAX_STEPS = 8

RESEARCHER_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the web. Returns up to max_results items.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"},
                    "recency": {"type": "string",
                                "enum": ["any", "day", "week", "month", "year"]},
                    "max_results": {"type": "integer"},
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "fetch_page",
            "description": "Fetch the full text of a URL.",
            "parameters": {
                "type": "object",
                "properties": {
                    "url": {"type": "string"},
                    "max_chars": {"type": "integer"},
                },
                "required": ["url"],
            },
        },
    },
]


def _researcher_execute(name: str, args: dict) -> dict:
    if name == "web_search":
        return web_search(args.get("query", ""), args.get("recency", "any"),
                          args.get("max_results", 8))
    if name == "fetch_page":
        return fetch_page(args.get("url", ""), args.get("max_chars", 8000))
    return {"status": "error", "kind": "unknown_tool", "detail": name}


RESEARCHER_SYSTEM_PROMPT = """You are a researcher worker. You receive ONE question.
Your job: search the web, fetch the most relevant 1-3 pages, and produce findings.

When you have enough to answer, emit your FINAL response as JSON only (no prose, no
markdown fences) in this shape:

  {"findings": "<2-4 sentences of factual claims with [1], [2] inline citations>",
   "citations": [{"url": "<url>", "title": "<title>"}, ...]}

Rules:
- Cite by [1], [2] inline; the citations list maps in order.
- Do NOT call the same tool with the same args twice.
- Do NOT produce prose outside the JSON envelope.
"""


def researcher_agent(question: str) -> dict:
    """Run a Lab 03-style research loop on a single question."""
    messages: list[dict] = [
        {"role": "system", "content": RESEARCHER_SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    seen: set[str] = set()
    fetches: list[dict] = []  # track ok fetches for partial-result reporting

    for _step in range(WORKER_MAX_STEPS):
        msg = chat_with_tools(messages, tools=RESEARCHER_TOOLS)
        entry: dict[str, Any] = {"role": "assistant", "content": msg.content}
        if msg.tool_calls:
            entry["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.name, "arguments": json.dumps(tc.arguments)}}
                for tc in msg.tool_calls
            ]
        messages.append(entry)

        if not msg.tool_calls:
            # Try to parse as final JSON envelope
            try:
                raw = (msg.content or "").strip()
                if raw.startswith("```"):
                    raw = raw.strip("`").split("\n", 1)[1].rstrip("`").strip()
                obj = json.loads(raw)
                return {"status": "ok",
                        "findings": obj.get("findings", ""),
                        "citations": obj.get("citations", [])}
            except (json.JSONDecodeError, IndexError):
                return {"status": "error", "kind": "bad_envelope",
                        "detail": "researcher returned non-JSON final"}

        for tc in msg.tool_calls:
            ah = _action_hash(tc.name, tc.arguments)
            if ah in seen:
                tool_result = {"status": "error", "kind": "repeated_action",
                               "detail": "you already called this with same args"}
            else:
                seen.add(ah)
                tool_result = _researcher_execute(tc.name, tc.arguments)
                if tc.name == "fetch_page" and tool_result.get("status") in ("ok", "too_long"):
                    fetches.append({"url": tool_result["url"],
                                    "title": tool_result.get("title", "")})
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(tool_result)[:4000]})

    # Step cap: surface partial findings honestly
    partial = f"Research did not complete within {WORKER_MAX_STEPS} steps."
    if fetches:
        partial += f" Fetched {len(fetches)} page(s) without producing a final summary."
    return {"status": "step_cap", "findings": partial, "citations": fetches}


## Writer worker

Takes the researcher's brief (findings + citations), composes ~150 words of cited prose. No tools. One LLM call.

In [ ]:
WRITER_SYSTEM_PROMPT = """You are a writer worker. You receive a brief containing
findings and a list of citations. Produce ~150 words of clean prose that:

1. States the findings accurately. Do not invent claims not present in the brief.
2. Preserves the citations. Reference them inline using [1], [2], etc. matching
   the order in the citation list. Then list the citations at the end as:

       [1] Title — URL
       [2] Title — URL

3. Does not add information from outside the brief.

If the brief is partial (status='step_cap'), say so explicitly: "Research did not
fully complete, but here is what was found..."

Return ONLY the prose. No JSON wrapping.
"""


def writer_agent(findings: str, citations: list[dict],
                  brief_status: str = "ok") -> dict:
    citation_lines = "\n".join(
        f"[{i + 1}] {c.get('title', '?')} — {c.get('url', '?')}"
        for i, c in enumerate(citations)
    )
    user_prompt = (
        f"BRIEF (status: {brief_status}):\n\n"
        f"FINDINGS:\n{findings}\n\n"
        f"CITATIONS:\n{citation_lines or '(none)'}\n\n"
        f"Compose the prose."
    )
    msg = chat_with_tools(
        [{"role": "system", "content": WRITER_SYSTEM_PROMPT},
         {"role": "user", "content": user_prompt}],
        temperature=0,
    )
    return {"status": "ok", "prose": (msg.content or "").strip()}


## Supervisor

The supervisor's tool registry exposes two worker-calls: `call_researcher` and `call_writer`. Strict Pydantic args reject LLM-invented fields. Action-hash dedup at the supervisor level prevents the supervisor from re-issuing the same worker call. Structured-error envelopes catch unknown workers and dispatch errors.

In [ ]:
SUPERVISOR_MAX_STEPS = 6


class CallResearcherArgs(StrictModel):
    question: str = Field(
        description="A clear, self-contained question for the researcher.",
    )


class CallWriterArgs(StrictModel):
    findings: str = Field(description="The researcher's findings text.")
    citations: list[dict] = Field(
        default_factory=list,
        description="The researcher's citation list as-is. Pass through verbatim.",
    )
    brief_status: str = Field(
        default="ok",
        description="The researcher's status — 'ok' or 'step_cap'.",
    )


def _call_researcher_tool(args: CallResearcherArgs) -> dict:
    return researcher_agent(args.question)


def _call_writer_tool(args: CallWriterArgs) -> dict:
    return writer_agent(args.findings, args.citations, args.brief_status)


SUPERVISOR_TOOLS_REGISTRY: dict = {
    "call_researcher": (
        _call_researcher_tool, CallResearcherArgs,
        "Dispatch a single question to the researcher worker. The researcher "
        "searches the web and returns {status, findings, citations}. Check "
        "status first — 'ok' means full results; 'step_cap' means partial.",
    ),
    "call_writer": (
        _call_writer_tool, CallWriterArgs,
        "Dispatch the researcher's brief to the writer worker. Pass findings "
        "and citations VERBATIM as you received them from the researcher. "
        "Returns {status, prose}.",
    ),
}


def _supervisor_schemas() -> list[dict]:
    return [
        {"type": "function",
         "function": {"name": name, "description": desc,
                      "parameters": args_model.model_json_schema()}}
        for name, (_fn, args_model, desc) in SUPERVISOR_TOOLS_REGISTRY.items()
    ]


def _supervisor_dispatch(call: ToolCall) -> dict:
    if call.name not in SUPERVISOR_TOOLS_REGISTRY:
        return {"status": "error", "kind": "unknown_worker", "tool": call.name,
                "available": list(SUPERVISOR_TOOLS_REGISTRY)}
    fn, args_model, _ = SUPERVISOR_TOOLS_REGISTRY[call.name]
    try:
        return fn(args_model.model_validate(call.arguments))
    except Exception as e:
        return {"status": "error", "kind": "supervisor_dispatch_error",
                "detail": f"{type(e).__name__}: {e}"}


SUPERVISOR_SYSTEM_PROMPT = """You are a supervisor agent. You coordinate two workers
via tool calls: call_researcher and call_writer.

WORKFLOW:
1. Call call_researcher with the user's question.
2. Read the researcher's return envelope:
   - {status: "ok"}: proceed to writer with findings + citations.
   - {status: "step_cap"}: still call the writer; pass brief_status="step_cap".
3. Call call_writer with the researcher's findings, citations, and status.
4. Return the writer's prose as your final answer (no JSON wrapping).

RULES:
- Do not call the same worker twice with the same args.
- Pass the researcher's citations to the writer VERBATIM — do not paraphrase or
  re-number them.
- If a worker returns an error envelope, surface the error rather than retrying.
"""


def supervisor_agent(task: str) -> dict:
    messages: list[dict] = [
        {"role": "system", "content": SUPERVISOR_SYSTEM_PROMPT},
        {"role": "user", "content": task},
    ]
    seen_actions: set[str] = set()
    schemas = _supervisor_schemas()

    for _step in range(SUPERVISOR_MAX_STEPS):
        msg = chat_with_tools(messages, tools=schemas)
        entry: dict[str, Any] = {"role": "assistant", "content": msg.content}
        if msg.tool_calls:
            entry["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.name, "arguments": json.dumps(tc.arguments)}}
                for tc in msg.tool_calls
            ]
        messages.append(entry)

        if not msg.tool_calls:
            return {"status": "ok", "answer": msg.content or "",
                    "steps_used": _step + 1}

        for tc in msg.tool_calls:
            ah = _action_hash(tc.name, tc.arguments)
            if ah in seen_actions:
                tool_result = {"status": "error", "kind": "repeated_action",
                               "detail": f"You already called {tc.name} with these args."}
            else:
                seen_actions.add(ah)
                tool_result = _supervisor_dispatch(tc)
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(tool_result)[:6000]})

    return {"status": "step_cap", "answer": "[supervisor hit step cap]",
            "steps_used": SUPERVISOR_MAX_STEPS}


## Demo

In [ ]:
task = (
    "Research recent developments in the Model Context Protocol (MCP) "
    "and write a 150-word summary."
)
result = supervisor_agent(task)
print(f"Status: {result['status']}, steps: {result['steps_used']}")
print("=" * 70)
print(result["answer"])


**Sample output (LLM responses will vary — live web; trajectory should be stable):**

```
Status: ok, steps: 3
======================================================================
The Model Context Protocol (MCP) is an open standard introduced by
Anthropic for connecting AI agents to external data sources and tools
[1]. Recent developments include expanded server libraries [2] and
production deployments at major companies [3]. The protocol uses a
client-server architecture where AI applications connect to MCP servers
that expose specific capabilities...

[1] Introducing the Model Context Protocol — https://www.anthropic.com/news/...
[2] MCP Server Gallery — https://...
[3] ...
```

Three supervisor steps: call researcher, call writer, finalize. Researcher's brief and citations flow through to writer verbatim. Final answer is the writer's prose.

## Production readiness — out of scope here

For a real deployment you'd also want: structured logging at every handoff (supervisor → worker → tool → return), trace IDs that span agent levels, per-worker cost budgets enforced by the supervisor, circuit breakers when a worker hits its cap repeatedly, eval harnesses that score citation preservation and step efficiency, and worker timeouts independent of LLM API timeouts. None of these change the canonical pattern — they're operational concerns layered on top.

The next pattern (Lab 11) adds a critic worker to this supervisor for iterative refinement. Same machinery; one new worker role.